<a href="https://colab.research.google.com/github/zk025815-beep/HCCDA-AI-Course/blob/main/RAG_with_langchain.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install langchain langchain-community pypdf

In [5]:
!pip install pymupdf

In [6]:
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader

# Yahan apni upload ki hui PDF file ka exact naam likhein
file_path = "/content/TLM-HCCDA AI V 2.0.pdf"

# Loader ko initialize karein
loader = PyMuPDFLoader(file_path)

# PDF ko load karein (Yeh har page ko ek alag Document bana dega)
pages = loader.load()

# Check karte hain PDF mein kitne pages hain
print(f"Is PDF mein total {len(pages)} pages hain.\n")

# Pichle step ka practical: Pehle page (Index 0) ka content aur metadata dekhein
print("=== PEHLE PAGE KA TEXT (page_content) ===")
# Text bohat lamba ho sakta hai isliye sirf pehle 500 characters print kar rahe hain
print(pages[0].page_content[0:500])
print("...\n")

print("=== PEHLE PAGE KA METADATA ===")
print(pages[0].metadata)

Is PDF mein total 15 pages hain.

=== PEHLE PAGE KA TEXT (page_content) ===
Government of Pakistan 
 
National Vocational and Technical Training Commission 
(NAVTTC) 
 
"Prime Minister Youth Skill Development Program" 
 
 
 
 
 
 
 
Course Contents / Lesson Plan 
Course Title: Huawei HCCDA-AI 
Duration: 3 Months
...

=== PEHLE PAGE KA METADATA ===
{'producer': 'Microsoft® Word LTSC', 'creator': 'Microsoft® Word LTSC', 'creationdate': '2025-05-29T15:22:45+05:00', 'source': '/content/TLM-HCCDA AI V 2.0.pdf', 'file_path': '/content/TLM-HCCDA AI V 2.0.pdf', 'total_pages': 15, 'format': 'PDF 1.7', 'title': '', 'author': 'Admin', 'subject': '', 'keywords': '', 'moddate': '2025-05-29T15:22:45+05:00', 'trapped': '', 'modDate': "D:20250529152245+05'00'", 'creationDate': "D:20250529152245+05'00'", 'page': 0}


In [7]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 1. Splitter ki settings define karein
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,    # Ek chunk mein maximum 1000 characters honge
    chunk_overlap=200,  # Pichle chunk ke 200 characters naye chunk mein shamil honge
    length_function=len # Length measure karne ke liye standard Python len function
)

# 2. PDF ke pages ko chunks mein split karein
chunks = text_splitter.split_documents(pages)

# 3. Results check karein
print(f"Original Pages ki tadad: {len(pages)}")
print(f"Chunks banne ke baad total tadad: {len(chunks)}\n")

# 4. Pehla chunk aur uska metadata dekhein
print("=== PEHLA CHUNK (page_content) ===")
print(chunks[0].page_content)

print("\n=== PEHLE CHUNK KA METADATA ===")
print(chunks[0].metadata)

Original Pages ki tadad: 15
Chunks banne ke baad total tadad: 29

=== PEHLA CHUNK (page_content) ===
Government of Pakistan 
 
National Vocational and Technical Training Commission 
(NAVTTC) 
 
"Prime Minister Youth Skill Development Program" 
 
 
 
 
 
 
 
Course Contents / Lesson Plan 
Course Title: Huawei HCCDA-AI 
Duration: 3 Months

=== PEHLE CHUNK KA METADATA ===
{'producer': 'Microsoft® Word LTSC', 'creator': 'Microsoft® Word LTSC', 'creationdate': '2025-05-29T15:22:45+05:00', 'source': '/content/TLM-HCCDA AI V 2.0.pdf', 'file_path': '/content/TLM-HCCDA AI V 2.0.pdf', 'total_pages': 15, 'format': 'PDF 1.7', 'title': '', 'author': 'Admin', 'subject': '', 'keywords': '', 'moddate': '2025-05-29T15:22:45+05:00', 'trapped': '', 'modDate': "D:20250529152245+05'00'", 'creationDate': "D:20250529152245+05'00'", 'page': 0}


In [8]:
!pip install sentence-transformers langchain-huggingface

In [10]:
from langchain_huggingface import HuggingFaceEmbeddings

# 1. HuggingFace ka embedding model initialize karein
embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# 2. Test karne ke liye apne pehle chunk ka text lein
text_sample = chunks[0].page_content

# 3. Text ko vector (numbers) mein convert karein
vector = embedding_model.embed_query(text_sample)

# 4. Results check karein
print(f"Is vector mein total dimensions (numbers): {len(vector)}\n")
print("Vector ke shuru ke 5 numbers dekhein:")
print(vector[:5])

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Is vector mein total dimensions (numbers): 384

Vector ke shuru ke 5 numbers dekhein:
[-0.03165004029870033, 0.052706144750118256, -0.01484854705631733, -0.02675168402493, -0.01908108778297901]


In [ ]:
# !pip install faiss-cpu

In [ ]:
# from langchain_community.vectorstores import FAISS
# from langchain_community.embeddings import HuggingFaceEmbeddings

# # 1. Apna LangChain wala embedding model initialize karein
# print("Embedding model load ho raha hai...")
# embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# # 2. Chunks aur Embeddings ko mila kar FAISS Vector Database banayen
# print("Vector Database ban rahi hai (is mein thoda time lag sakta hai)...")
# vector_db = FAISS.from_documents(chunks, embedding_model)
# print("Database successfully ban gayi!\n")

# # ==========================================
# # 3. TESTING (Similarity Search)
# # ==========================================
# # Ab hum database se koi aisi baat poochenge jo aapki PDF mein likhi ho
# user_query = "who offer this course" # Yahan apni marzi ka sawal likhein

# # Database mein user ke sawal se milte julte top 3 chunks dhoondein
# matched_docs = vector_db.similarity_search(user_query, k=3)

# print(f"Sawal: {user_query}")
# print("-" * 40)
# print("Top Matched Result se mila hua text:\n")
# print(matched_docs[0].page_content)

In [11]:
!pip install chromadb langchain-chroma

In [12]:
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

# 1. Apna LangChain wala embedding model initialize karein
print("Embedding model load ho raha hai...")
embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# 2. ChromaDB banayen aur local folder mein save karein
print("ChromaDB Vector Store ban raha hai...")
persist_dir = "./my_chroma_db" # Is folder mein aapki database save hogi

vector_db = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    persist_directory=persist_dir
)
print(f"Database successfully ban gayi aur '{persist_dir}' folder mein save ho gayi!\n")

# ==========================================
# 3. TESTING (Similarity Search)
# ==========================================
user_query = "Duration os this course?" # Yahan apni marzi ka sawal likhein

# Database se top 3 relevant chunks dhoondein
matched_docs = vector_db.similarity_search(user_query, k=3)

print(f"Sawal: {user_query}")
print("-" * 40)
print("Top Matched Result ka text:\n")
print(matched_docs[0].page_content)

Embedding model load ho raha hai...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

ChromaDB Vector Store ban raha hai...
Database successfully ban gayi aur './my_chroma_db' folder mein save ho gayi!

Sawal: Duration os this course?
----------------------------------------
Top Matched Result ka text:

Government of Pakistan 
 
National Vocational and Technical Training Commission 
(NAVTTC) 
 
"Prime Minister Youth Skill Development Program" 
 
 
 
 
 
 
 
Course Contents / Lesson Plan 
Course Title: Huawei HCCDA-AI 
Duration: 3 Months


In [13]:
!pip install langchain-groq

In [14]:
!pip install langchain langchain-community langchain-core langchain-groq langchain-chroma

In [41]:
import os
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from google.colab import userdata

# 1. Retrieve your Groq API key securely
try:
    my_groq_key = userdata.get('GROK_API_KEY')
except Exception:
    my_groq_key = userdata.get('GROQ_API_KEY')

# Get the active model from current notebook state variable if available, else fallback
try:
    selected_model = active_chat_models[0]
except NameError:
    selected_model = "openai/gpt-oss-120b"

print(f"Using active model: {selected_model}")

# 2. LLM Initialize using an active model from your list
llm = ChatGroq(
    groq_api_key=my_groq_key,
    model_name=selected_model,
    temperature=0
)

retriever = vector_db.as_retriever(search_kwargs={"k": 3})

prompt = ChatPromptTemplate.from_template("""
You are a helpful AI assistant. Answer the user's question strictly based on the provided context below.
Always give your answer in Simple English (or Roman Urdu using English alphabets). Do NOT use Hindi/Devanagari script.
If the answer is not in the context, simply say "I could not find the answer in the document."

Context:
{context}

Question: {question}

Answer:
""")

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

# 3. Final Test (Sirf jawab print hoga)
response = rag_chain.invoke("what i will become after this course?")
print("=== FINAL AI JAWAB ===")
print(response)

Using active model: openai/gpt-oss-120b
=== FINAL AI JAWAB ===
After finishing this program you will be a qualified AI trainer/educator.  
You will be able to design and give technical AI training, assess learners, mentor them, and work on real‑world AI projects such as chat‑bots, smart‑retail or inspection systems.
